# Altınkaynak Fiyat Servisi API

**Altınkaynak** statik fiyat feed'i — döviz + altın/gümüş anlık **özet** ve **detay**.

> Public endpoint, auth yok. CDN cache'li — güncelleme ~60 sn.

## Endpoint

```
GET https://static.altinkaynak.com/Store_Main
GET https://static.altinkaynak.com/Store_Gold
```

**Header:** `accept: application/json` · **Auth:** yok · **Cache:** CDN · **Güncelleme:** ~60 sn

## Store_Main · Özet (7 kayıt)

`{ generatedAt, items[] }` — döviz + öne çıkan altın özeti.

### Yanıt alanları

| Alan | Açıklama |
|------|----------|
| `generatedAt` | Feed üretim zamanı · ISO 8601 (UTC) |
| `items[]` | 7 elemanlı özet dizi · döviz + öne çıkan altın |
| `code` | Kayıt kodu (USD, EUR, HH_T, XAUUSD, ...) |
| `value` | Satış fiyatı · ana gösterim (string, TR format) |
| `buy` / `sell` | Alış / satış · string (XAUUSD hariç float) |
| `time` | Kaydın kendi güncelleme zamanı (ISO 8601) |
| `IsBulk` | Toptan mı · `true` / `false` / `null` (XAUUSD: null) |
| `change` | Günlük değişim yüzdesi · float |

### items[] kodları

| Kod | Anlam |
|-----|-------|
| `USD` | ABD Doları |
| `EUR` | Euro |
| `HH_T` | Has Altın Toptan · 24 ayar |
| `CH_T` | Cumhuriyet Altın Toptan · 24 ayar |
| `A` | Ata Cumhuriyet Altını |
| `C` | Çeyrek Altın |
| `XAUUSD` | Ons (USD) · spot |

## Store_Gold · Detay (24 kayıt)

`Array<{...}>` → 24 kayıt · `DataGroup` ile kategori filtrelenir.

### Her kayıtta dönen alanlar

| Alan | Örnek |
|------|-------|
| `Id` | `20` (sayısal ID) |
| `Kod` | `"HH_T"` (kayıt kodu) |
| `Alis` / `Satis` | `"6.737,92"` / `"6.787,08"` (string, TR format) |
| `Aciklama` | `"24 Ayar Saf Altın (saflık 0.9999)"` |
| `MobilAciklama` | `"Has Toptan"` (mobil için kısa) |
| `WidgetAciklama` | `"Has T"` (widget için ultra kısa) |
| `GuncellenmeZamani` | `"24.04.2026 10:04:47"` (TR format) |
| `Main` | `false` (tüm kayıtlarda false) |
| `DataGroup` | `11` (kategori kodu · 2, 7, 8, 9, 11) |
| `WebGroup` | `null` · XAUUSD & IAB_KAPANIS için `9` |
| `IsBulk` | `true` / `false` (toptan mı) |
| `Change` | `-0.36` (günlük %, float) |

### DataGroup kategorileri

| DG | Kategori |
|----|----------|
| `2` | Perakende · 22/18/14 ayar + gram → `B`, `18`, `14`, `GA` |
| `7` | Gümüş toptan → `AG_T` |
| `8` | Cumhuriyet ailesi → `A`, `A_T`, `A5`, `C`, `Y`, `T`, `G`, `R`, `H` |
| `9` | Kullanılmış + ONS + IAB → `EC`, `EY`, `ET`, `EG`, `XAUUSD`, `IAB_KAPANIS` |
| `11` | Toptan altın (24/22 ayar) → `HH_T`, `CH_T`, `GAT`, `B_T` |

## 1. Ortak Ayarlar

In [6]:
import requests

MAIN_URL = "https://static.altinkaynak.com/Store_Main"
GOLD_URL = "https://static.altinkaynak.com/Store_Gold"

HEADERS = {"accept": "application/json"}


def trf(s):
    """TR formatlı string'i ("6.662,88") float'a çevir. Sayı zaten float ise dokunma."""
    if isinstance(s, (int, float)):
        return float(s)
    return float(s.replace(".", "").replace(",", "."))

## 2. Store_Main · Özet (döviz + öne çıkan altın)

7 elemanlı özet feed: USD, EUR, HH_T, CH_T, A, C, XAUUSD.

In [7]:
r = requests.get(MAIN_URL, headers=HEADERS)
r.raise_for_status()
main = r.json()

print(f"Üretim: {main['generatedAt']}  ·  {len(main['items'])} kayıt\n")
print(f"{'Kod':<8} {'Alış':>12} {'Satış':>12} {'Değişim':>10}  {'Toptan':<7} {'Saat':<19}")
print("-" * 78)
for it in main["items"]:
    bulk = {True: "evet", False: "hayır", None: "-"}[it["IsBulk"]]
    saat = it["time"][:19].replace("T", " ")
    print(f"{it['code']:<8} {trf(it['buy']):>12.4f} {trf(it['sell']):>12.4f} {it['change']:>9.2f}%  {bulk:<7} {saat}")

Üretim: 2026-04-28T13:43:29.342516  ·  7 kayıt

Kod              Alış        Satış    Değişim  Toptan  Saat               
------------------------------------------------------------------------------
USD           44.9000      45.1010      0.31%  hayır   2026-04-28 16:43:21
EUR           52.4690      52.7830      0.23%  hayır   2026-04-28 16:43:21
HH_T        6617.3400    6669.2300     -1.17%  evet    2026-04-28 16:43:21
CH_T        6584.2500    6635.8800     -1.17%  evet    2026-04-28 16:43:21
A          43898.2300   48000.0000     -3.03%  hayır   2026-04-28 16:43:21
C          10717.4800   11650.0000     -2.10%  hayır   2026-04-28 16:43:21
XAUUSD      4590.4800    4590.4800      0.00%  -       2026-04-28 16:43:12


## 3. Store_Gold · Detay (24 kayıt, DataGroup'a göre gruplu)

Tüm altın/gümüş ürünleri tek dizi olarak gelir; `DataGroup` ile kategori filtrelenir.

In [8]:
from collections import defaultdict

DG_LABEL = {
    2: "Perakende (22/18/14 ayar + gram)",
    7: "Gümüş toptan",
    8: "Cumhuriyet ailesi",
    9: "Kullanılmış + ONS + IAB",
    11: "Toptan altın (24/22 ayar)",
}

r = requests.get(GOLD_URL, headers=HEADERS)
r.raise_for_status()
gold = r.json()

by_dg = defaultdict(list)
for row in gold:
    by_dg[row["DataGroup"]].append(row)

print(f"Toplam: {len(gold)} kayıt · {len(by_dg)} kategori\n")
for dg in sorted(by_dg):
    print(f"=== DG={dg} · {DG_LABEL.get(dg, '?')} ({len(by_dg[dg])} kayıt) ===")
    print(f"  {'Kod':<14} {'Alış':>12} {'Satış':>12} {'Değişim':>10}  Açıklama")
    print("  " + "-" * 78)
    for row in by_dg[dg]:
        chg = f"{row['Change']:>9.2f}%" if row['Change'] is not None else f"{'-':>10}"
        print(f"  {row['Kod']:<14} {trf(row['Alis']):>12.4f} {trf(row['Satis']):>12.4f} {chg}  {row['Aciklama']}")
    print()

Toplam: 24 kayıt · 5 kategori

=== DG=2 · Perakende (22/18/14 ayar + gram) (4 kayıt) ===
  Kod                    Alış        Satış    Değişim  Açıklama
  ------------------------------------------------------------------------------
  B                 6034.8100    6470.0000     -2.85%  22 Ayar Altın  (Saflık Derecesi 0.916)
  18                4630.4800    5360.0000     -1.65%  18 Ayar Altın  (Saflık Derecesi 0.750)
  14                3748.8200    4220.0000     -1.86%  14 Ayar Altın  (Saflık Derecesi 0.585)
  GA                6584.2500    6740.0000     -1.89%  24 Ayar Saf Altın (saflık Derecesi 0.995) 1 GR

=== DG=7 · Gümüş toptan (1 kayıt) ===
  Kod                    Alış        Satış    Değişim  Açıklama
  ------------------------------------------------------------------------------
  AG_T                96.3400     109.6700     -1.70%  Gümüş (Saflık Derecesi 0.9999)

=== DG=8 · Cumhuriyet ailesi (9 kayıt) ===
  Kod                    Alış        Satış    Değişim  Açıklama
  --

## 4. Tek Kategori · DataGroup ile Filtreleme

Sadece toptan altın (DG=11) listesi.

In [9]:
toptan = [r for r in gold if r["DataGroup"] == 11]

print(f"Toptan altın · {len(toptan)} kayıt\n")
print(f"{'Kod':<8} {'Alış':>12} {'Satış':>12} {'Değişim':>10}  {'Güncelleme':<19}  Açıklama")
print("-" * 90)
for row in toptan:
    print(f"{row['Kod']:<8} {trf(row['Alis']):>12.4f} {trf(row['Satis']):>12.4f} {row['Change']:>9.2f}%  {row['GuncellenmeZamani']:<19}  {row['Aciklama']}")

Toptan altın · 4 kayıt

Kod              Alış        Satış    Değişim  Güncelleme           Açıklama
------------------------------------------------------------------------------------------
HH_T        6617.3400    6669.2300     -1.17%  28.04.2026 16:43:21  24 Ayar Saf Altın (saflık Derecesi 0.9999)
CH_T        6584.2500    6635.8800     -1.17%  28.04.2026 16:43:21  24 Ayar saf Altın (saflık Derecesi 0.995)
GAT         6584.2500    6712.1900     -1.17%  28.04.2026 16:43:21  24 Ayar Saf Altın (Saflık Derecesi 0.995)
B_T         6034.8100    6082.7400     -1.17%  28.04.2026 16:43:21  22 Ayar Altın (Saflık Derecesi 0.916)


## 5. curl Eşdeğeri

`jq` + `column` ile okunabilir çıktı.

In [10]:
%%bash
# Store_Main → döviz + özet altın (7 kayıt)
curl -s "https://static.altinkaynak.com/Store_Main" \
  | jq -r '.items[] | [.code, .buy, .sell, "\(.change)%"] | @tsv' \
  | column -t -s $'\t'

echo

# Store_Gold → detay (24 kayıt) · DataGroup'a göre grupla
curl -s "https://static.altinkaynak.com/Store_Gold" \
  | jq -r 'group_by(.DataGroup)[]
      | "=== DG=\(.[0].DataGroup) ===",
        (.[] | "  \(.Kod)\t\(.Alis)/\(.Satis)\t\(.Change)%\t\(.Aciklama)")'

USD     44,900     45,102     0.31%
EUR     52,467     52,781     0.23%
HH_T    6.626,89   6.678,90   -1.03%
CH_T    6.593,76   6.645,51   -1.03%
A       43.961,62  48.000,00  -3.03%
C       10.732,96  11.650,00  -2.1%
XAUUSD  4589.88    4589.88    0.0%

=== DG=2 ===
  B	6.043,53/6.470,00	-2.85%	22 Ayar Altın  (Saflık Derecesi 0.916)
  18	4.637,17/5.360,00	-1.65%	18 Ayar Altın  (Saflık Derecesi 0.750)
  14	3.754,23/4.220,00	-1.86%	14 Ayar Altın  (Saflık Derecesi 0.585)
  GA	6.593,76/6.740,00	-1.89%	24 Ayar Saf Altın (saflık Derecesi 0.995) 1 GR
=== DG=7 ===
  AG_T	96,51/109,86	-1.52%	Gümüş (Saflık Derecesi 0.9999)
=== DG=8 ===
  A	43.961,62/48.000,00	-3.03%	22 Ayar Altın  (Saflık Derecesi 0.916) 7.20 GR
  A_T	43.961,62/45.295,85	-1.03%	22 Ayar Altın (Saflık Derecesi 0.916) 7.20 GR
  C	10.732,96/11.650,00	-2.1%	22 Ayar Altın  (Saflık Derecesi 0.916) 1.75 GR
  Y	21.323,39/23.300,00	-2.1%	22 Ayar Altın  (Saflık Derecesi 0.916) 3.50 GR
  T	42.476,59/46.600,00	-2.1%	22 Ayar Altın  (Saflık D

## Notlar

- **TR sayı formatı:** Store_Main'de `value`/`buy`/`sell` ve Store_Gold'da `Alis`/`Satis` string olarak `"6.662,88"` döner — sayısal işlem için `trf()` ile float'a çevir.
- **XAUUSD istisnası:** Store_Main'de `XAUUSD` kaydı için `buy`/`sell` zaten float ve `IsBulk` `null` gelir.
- **`Change` null olabilir:** Store_Gold'da DG=9 altındaki `XAUUSD` ve `IAB_KAPANIS` kayıtlarında `Change` `null` döner — format öncesi kontrol et.
- **Zaman formatı farkı:** Store_Main `time` ISO 8601 (`2026-04-28T16:35:21.679394`), Store_Gold `GuncellenmeZamani` TR format (`28.04.2026 16:35:31`).
- **`generatedAt`:** sadece Store_Main'de var; Store_Gold için kayıt başına `GuncellenmeZamani` kullan.
- **Kategori filtresi:** API tarafında filtre yok — `DataGroup` üzerinden client tarafında filtrele.
- **Cache:** CDN cache'li, ~60 sn'de bir tazelenir; daha sık polling anlamsız.
- **Auth yok:** `accept: application/json` dışında header gerekmez.